In [18]:
import asyncio
from dataclasses import dataclass
from typing import Any, Literal, List
from google import genai
import pandas as pd

from pydantic import BaseModel, Field
from pydantic_ai import Agent

import nest_asyncio

from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

d:\e\Devina\Review Thing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df_reviews=pd.read_json('Dataset/all_reviews.json', orient="records")
df_reviews

,id,item_id,review_time,variant,translated_text,polarity_score,sentiment,accu_polarity_score
0,5c4aebb3-c873-4303-ba32-12b67b20dabf,20bcb1c5-1b2b-46e4-9586-db1fa0d7d371,2022-09-05 21:55:00,Variation: BLACK,Just arrived ....Thank God for good condition.,"{'Neutral': 0.15824031830000002, 'Negative': 0...",Positive,1.662962
1,df9eb09d-a456-4d8d-ac08-9a879e4a0229,20bcb1c5-1b2b-46e4-9586-db1fa0d7d371,2023-01-17 18:43:00,Variation: BLACK,Material: Lightweight plastic bags Durability:...,"{'Neutral': 0.5387101769, 'Negative': 0.021078...",Neutral,1.376977
2,aae9fc51-0645-46aa-b729-91dafb5e7ac5,20bcb1c5-1b2b-46e4-9586-db1fa0d7d371,2022-10-15 13:45:00,Variation: WHITE,Material: Ok ok at the price of Durability: Ok...,"{'Neutral': 0.71989429, 'Negative': 0.16190171...",Neutral,0.632499
3,5579413f-ef98-4285-974e-bc36e45545f2,20bcb1c5-1b2b-46e4-9586-db1fa0d7d371,2022-12-04 21:18:00,Variation: PINK,Material: Waterproof Durability: Okay nice bea...,"{'Neutral': 0.3992878795, 'Negative': 0.025619...",Positive,1.498233
4,beec2f20-5917-4088-86f1-911b5af4ea4e,20bcb1c5-1b2b-46e4-9586-db1fa0d7d371,2022-09-17 10:32:00,Variation: BLACK,My son is so excited and excited to be able to...,"{'Neutral': 0.0717492327, 'Negative': 0.015812...",Positive,1.865002
...,...,...,...,...,...,...,...,...
24608,046dcd41-f347-4cbe-8cfd-ba72ab21831f,5647c11b-1c08-4408-a492-bf3fcb77405b,2022-02-11 21:51:00,Variation: Navy,Welcome to.Postony speed.Best.Pretty.Love.Than...,"{'Neutral': 0.07302926480000001, 'Negative': 0...",Positive,1.816420
24609,06f12a41-1049-4266-bd27-3ea72977fbbf,5647c11b-1c08-4408-a492-bf3fcb77405b,2021-11-18 14:58:00,Variation: Navy,"Put wallets, phones, and other items, perfect ...","{'Neutral': 0.200163573, 'Negative': 0.0614048...",Positive,1.554217
24610,166f3338-10c0-4f72-9062-5e8eacbc6268,5647c11b-1c08-4408-a492-bf3fcb77405b,2022-10-03 16:12:00,Variation: Black,Quality is good.Got a few compartments.Zip is ...,"{'Neutral': 0.1136623919, 'Negative': 0.027559...",Positive,1.776099
24611,03a9a943-bff9-4994-9df7-288a2ffbb59e,5647c11b-1c08-4408-a492-bf3fcb77405b,2022-03-31 20:23:00,Variation: Black,Good product.Quality.Only one buckle sling is ...,"{'Neutral': 0.2263131291, 'Negative': 0.199059...",Positive,0.977450


In [5]:
df_items=pd.read_json("Dataset/all_items.json",orient="records")
df_items

,id,name,ratings,ratings_num,sold,shop_name,url,description,price,accu_score,image
0,20bcb1c5-1b2b-46e4-9586-db1fa0d7d371,NIKE Fashion TravelSchool Backpack Bag For Unisex,4.8,162,447,sunsuncps,https://shopee.com.my/NIKE-Fashion-Travel-Scho...,High quality Pre-order For Unisex Fashion Bag ...,RM38.00,1.061794,https://down-my.img.susercontent.com/file/65b2...
1,988bec91-c0a1-41e8-97b4-70e324843079,Wallet For Men PU Leather Card Holder Wallet R...,4.8,8100,17800,mtfashion2.my,https://shopee.com.my/Wallet-For-Men-PU-Leathe...,MTFASHION MALAYSIA READY STOCK Ship with...,RM9.96 - RM28.99,1.218403,https://down-my.img.susercontent.com/file/662c...
2,27973f6a-ddc8-4546-a4d4-5458bffef04b,READY STOCK ❤️ Premium 914 Inch Laptop Bag Pou...,4.9,789,2000,junhomedecor,https://shopee.com.my/READY-STOCK-❤️-Premium-9...,Welcome to our fun little store All products a...,RM24.40 - RM26.40,1.359126,https://down-my.img.susercontent.com/file/9a45...
3,0c117e89-4ba7-49d4-8fe3-d2c6ea0dcfee,CS New Men Leather Messenger Bag Single Should...,4.9,934,1800,chooseset.my,https://shopee.com.my/CS-New-Men-Leather-Messe...,Welcome to chooseset • Here is some FYI fr...,RM36.80,1.314783,https://down-my.img.susercontent.com/file/1a59...
4,a3dd88b3-e6b2-441d-ae19-e0014ba1d3ad,Ready Stock Nike Backpack Bag Nike Unisex Air ...,4.8,788,2000,handbagmurah.,https://shopee.com.my/Ready-Stock-Nike-Backpac...,ALL READY STOCK MALAYSIA Local Sellers Malay...,RM26.90 - RM30.90,1.354279,https://down-my.img.susercontent.com/file/sg-1...
...,...,...,...,...,...,...,...,...,...,...,...
227,6077bfd6-4198-4cca-a528-4e97eadbe460,Messenger Bag Water Resistant Beg Lelaki Bag ...,4.9,546,1100,jstarcollection,https://shopee.com.my/Messenger-Bag-Water-Resi...,READY STOCK MALAYSIA FAST SHIPPING Item Type ...,RM23.21,1.289334,https://down-my.img.susercontent.com/file/fd7f...
228,284b1a31-2c49-4c4a-9156-87dce6516227,Beg Galas Lelaki Beg Galas Belakang Canvas Sch...,4.8,2100,5100,emakbaba,https://shopee.com.my/Beg-Galas-Lelaki-Beg-Gal...,Beg galas belakang canvas school backpack Fab...,RM13.85,1.209899,https://down-my.img.susercontent.com/file/0d2e...
229,c73577ed-5946-4de3-854b-49be7c26a48d,MenBense Mens Short New Wallet MultiCard Mens ...,4.8,2400,6100,ibestbuy,https://shopee.com.my/MenBense-Men's-Short-New...,MenBense Mens Short New Wallet Fashion Casual ...,RM9.90,1.362291,https://down-my.img.susercontent.com/file/9cf5...
230,2eb0ca6b-79e5-416d-a13f-79c26697ad9d,PLAYBOY Men Short Wallets Zipper Leather Purse...,4.7,1100,3300,jianghao1.my,https://shopee.com.my/PLAYBOY-Men-Short-Wallet...,Spot Super Bargain\r Colour can be choose are ...,RM6.99,0.933272,https://down-my.img.susercontent.com/file/469b...


In [10]:
# merge df_reviews and df_items
df_combined=pd.merge(df_items,df_reviews,left_on="id",right_on="item_id",how="left")

df_combined=df_combined.drop(columns=["item_id"])
df_combined=df_combined.rename(columns={"id_x":"item_id","id_y":"review_id"})
df_combined

,item_id,name,ratings,ratings_num,sold,shop_name,url,description,price,accu_score,image,review_id,review_time,variant,translated_text,polarity_score,sentiment,accu_polarity_score
0,20bcb1c5-1b2b-46e4-9586-db1fa0d7d371,NIKE Fashion TravelSchool Backpack Bag For Unisex,4.8,162,447,sunsuncps,https://shopee.com.my/NIKE-Fashion-Travel-Scho...,High quality Pre-order For Unisex Fashion Bag ...,RM38.00,1.061794,https://down-my.img.susercontent.com/file/65b2...,5c4aebb3-c873-4303-ba32-12b67b20dabf,2022-09-05 21:55:00,Variation: BLACK,Just arrived ....Thank God for good condition.,"{'Neutral': 0.15824031830000002, 'Negative': 0...",Positive,1.662962
1,20bcb1c5-1b2b-46e4-9586-db1fa0d7d371,NIKE Fashion TravelSchool Backpack Bag For Unisex,4.8,162,447,sunsuncps,https://shopee.com.my/NIKE-Fashion-Travel-Scho...,High quality Pre-order For Unisex Fashion Bag ...,RM38.00,1.061794,https://down-my.img.susercontent.com/file/65b2...,df9eb09d-a456-4d8d-ac08-9a879e4a0229,2023-01-17 18:43:00,Variation: BLACK,Material: Lightweight plastic bags Durability:...,"{'Neutral': 0.5387101769, 'Negative': 0.021078...",Neutral,1.376977
2,20bcb1c5-1b2b-46e4-9586-db1fa0d7d371,NIKE Fashion TravelSchool Backpack Bag For Unisex,4.8,162,447,sunsuncps,https://shopee.com.my/NIKE-Fashion-Travel-Scho...,High quality Pre-order For Unisex Fashion Bag ...,RM38.00,1.061794,https://down-my.img.susercontent.com/file/65b2...,aae9fc51-0645-46aa-b729-91dafb5e7ac5,2022-10-15 13:45:00,Variation: WHITE,Material: Ok ok at the price of Durability: Ok...,"{'Neutral': 0.71989429, 'Negative': 0.16190171...",Neutral,0.632499
3,20bcb1c5-1b2b-46e4-9586-db1fa0d7d371,NIKE Fashion TravelSchool Backpack Bag For Unisex,4.8,162,447,sunsuncps,https://shopee.com.my/NIKE-Fashion-Travel-Scho...,High quality Pre-order For Unisex Fashion Bag ...,RM38.00,1.061794,https://down-my.img.susercontent.com/file/65b2...,5579413f-ef98-4285-974e-bc36e45545f2,2022-12-04 21:18:00,Variation: PINK,Material: Waterproof Durability: Okay nice bea...,"{'Neutral': 0.3992878795, 'Negative': 0.025619...",Positive,1.498233
4,20bcb1c5-1b2b-46e4-9586-db1fa0d7d371,NIKE Fashion TravelSchool Backpack Bag For Unisex,4.8,162,447,sunsuncps,https://shopee.com.my/NIKE-Fashion-Travel-Scho...,High quality Pre-order For Unisex Fashion Bag ...,RM38.00,1.061794,https://down-my.img.susercontent.com/file/65b2...,beec2f20-5917-4088-86f1-911b5af4ea4e,2022-09-17 10:32:00,Variation: BLACK,My son is so excited and excited to be able to...,"{'Neutral': 0.0717492327, 'Negative': 0.015812...",Positive,1.865002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24608,5647c11b-1c08-4408-a492-bf3fcb77405b,PRIA Rev Clutchbag Men Tas 2in Men Women Tas...,4.9,2700,5200,rev_store.my,https://shopee.com.my/PRIA-Rev-Clutchbag-Men-T...,Antproject⚡3 0 2 Extra Maximum Capacity Slingb...,RM17.90,1.300533,https://down-my.img.susercontent.com/file/811e...,046dcd41-f347-4cbe-8cfd-ba72ab21831f,2022-02-11 21:51:00,Variation: Navy,Welcome to.Postony speed.Best.Pretty.Love.Than...,"{'Neutral': 0.07302926480000001, 'Negative': 0...",Positive,1.816420
24609,5647c11b-1c08-4408-a492-bf3fcb77405b,PRIA Rev Clutchbag Men Tas 2in Men Women Tas...,4.9,2700,5200,rev_store.my,https://shopee.com.my/PRIA-Rev-Clutchbag-Men-T...,Antproject⚡3 0 2 Extra Maximum Capacity Slingb...,RM17.90,1.300533,https://down-my.img.susercontent.com/file/811e...,06f12a41-1049-4266-bd27-3ea72977fbbf,2021-11-18 14:58:00,Variation: Navy,"Put wallets, phones, and other items, perfect ...","{'Neutral': 0.200163573, 'Negative': 0.0614048...",Positive,1.554217
24610,5647c11b-1c08-4408-a492-bf3fcb77405b,PRIA Rev Clutchbag Men Tas 2in Men Women Tas...,4.9,2700,5200,rev_store.my,https://shopee.com.my/PRIA-Rev-Clutchbag-Men-T...,Antproject⚡3 0 2 Extra Maximum Capacity Slingb...,RM17.90,1.300533,https://down-my.img.susercontent.com/file/811e...,166f3338-10c0-4f72-9062-5e8eacbc6268,2022-10-03 16:12:00,Variation: Black,Quality is good.Got a few compartments.Zip is ...,"{'Neutral': 0.113662391

In [6]:
df_reviews.dtypes

id                                str
item_id                           str
review_time            datetime64[ms]
variant                           str
translated_text                   str
polarity_score                 object
sentiment                         str
accu_polarity_score           float64
dtype: object

# Batch Sentiment Analysis
We want to compare methods in which we can get the best sentiment analysis results.

In [ ]:
# define class
class AspectScore(BaseModel):
    aspect: str =   

In [ ]:
# Get aspect scores
# Aspect taxonomy: material quality, value, performance, shipping/delivery
# Berttopic + pydantic_ai

df_review_sample=df_combined.iloc[0:20]
docs=df_review_sample["translated_text"].tolist()

topic_model = BERTopic(calculate_probabilities=True)
topics, probs = topic_model.fit_transform(docs)

potential_topics = [i for i, p in enumerate(probs[0]) if p > 0.15]

potential_topics

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15262.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[1]

In [29]:
topics

[1,
 -1,
 0,
 -1,
 -1,
 1,
 1,
 0,
 -1,
 0,
 0,
 0,
 -1,
 -1,
 1,
 1,
 1,
 1,
 0,
 -1,
 0,
 -1,
 1,
 -1,
 0,
 0,
 0,
 0,
 -1,
 0,
 -1,
 1,
 -1,
 -1,
 -1,
 -1,
 1,
 -1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 -1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 -1,
 0,
 0,
 0,
 0,
 -1,
 0,
 -1,
 0,
 -1,
 -1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 -1,
 0,
 0,
 1,
 0,
 1,
 -1,
 0,
 0,
 1,
 -1,
 0,
 0,
 -1]